# Lab 13 Part 2 Search Functions and Retrieval Approaches

## Setup

In [1]:
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

documents = [
    "Transformers use self-attention to understand relationships between words.",
    "TF-IDF represents documents using word importance scores.",
    "Word2Vec learns dense vector representations of words.",
    "Named entity recognition identifies names, places, and organizations in text.",
    "Information retrieval finds relevant documents for a user query.",
    "Cosine similarity measures how similar two text vectors are."
]
vectorizer = TfidfVectorizer(stop_words='english')
document_vectors = vectorizer.fit_transform(documents)
print('Documents:', len(documents))
print('TF-IDF matrix:', document_vectors.shape)

Documents: 6
TF-IDF matrix: (6, 39)


## 1 Make a Simple Search Function

In [2]:
def ranked_results(query, k=2, positive_only=False):
    if not isinstance(query, str):
        raise TypeError('The query must be a string.')
    if isinstance(k, bool) or not isinstance(k, (int, np.integer)) or k < 1:
        raise ValueError('k must be a positive integer.')
    if document_vectors.shape[0] != len(documents):
        raise ValueError('Rebuild the document vectors after changing the collection.')
    query_vector = vectorizer.transform([query])
    scores = cosine_similarity(query_vector, document_vectors)[0]
    ranked_ids = np.lexsort((np.arange(len(documents)), -scores))
    if positive_only:
        ranked_ids = ranked_ids[scores[ranked_ids] > 0]
    return [dict(document_id=int(i), score=float(scores[i]), text=documents[i])
            for i in ranked_ids[:k]]

def search(query, k=2):
    results = ranked_results(query, k)
    print(f'Query: {query!r} | k={k}')
    for result in results:
        print(f"Document {result['document_id']} | Score: {result['score']:.3f} | {result['text']}")
    return results

similarity_results = search('How can we measure similarity between text vectors?')

Query: 'How can we measure similarity between text vectors?' | k=2
Document 5 | Score: 0.686 | Cosine similarity measures how similar two text vectors are.
Document 3 | Score: 0.148 | Named entity recognition identifies names, places, and organizations in text.


## 2 Sparse vs Dense Retrieval

TF-IDF matches weighted vocabulary terms; dense retrieval uses learned embeddings to capture semantic similarity. Hybrid retrieval combines both.

## 3 High Rank vs Good Match

In [3]:
unrelated_results = search('volcano eruption', k=2)
print('--- Relevant query ---')
relevant_results = search('cosine similarity', k=2)

Query: 'volcano eruption' | k=2
Document 0 | Score: 0.000 | Transformers use self-attention to understand relationships between words.
Document 1 | Score: 0.000 | TF-IDF represents documents using word importance scores.
--- Relevant query ---
Query: 'cosine similarity' | k=2
Document 5 | Score: 0.594 | Cosine similarity measures how similar two text vectors are.
Document 0 | Score: 0.000 | Transformers use self-attention to understand relationships between words.


### Brief Answer

All “volcano eruption” scores are zero, so its returned ranks are ties, ordered here by document ID. A top rank alone does not establish relevance.

## 4 Skip Results with Zero Similarity

In [4]:
def search_filtered(query, k=2):
    results = ranked_results(query, k, positive_only=True)
    print(f'Query: {query!r} | k={k}')
    if not results:
        print('No matching documents found.')
    for result in results:
        print(f"Document {result['document_id']} | Score: {result['score']:.3f} | {result['text']}")
    return results

filtered_unrelated = search_filtered('volcano eruption')
print('---')
filtered_relevant = search_filtered('cosine similarity')

Query: 'volcano eruption' | k=2
No matching documents found.
---
Query: 'cosine similarity' | k=2
Document 5 | Score: 0.594 | Cosine similarity measures how similar two text vectors are.


## 5 Practice Questions

### Question 1 Named Entity Recognition

In [5]:
ner_results = search('named entity recognition', k=2)
print('First-ranked document:', ner_results[0]['document_id'])

Query: 'named entity recognition' | k=2
Document 3 | Score: 0.625 | Named entity recognition identifies names, places, and organizations in text.
Document 0 | Score: 0.000 | Transformers use self-attention to understand relationships between words.
First-ranked document: 3


### Question 1 Answer

Document **3** ranks first: “Named entity recognition identifies names, places, and organizations in text.”

### Question 2 Add BERT and Rebuild the Index

In [6]:
bert_document = 'BERT is a transformer model that uses bidirectional context.'
old_shape = document_vectors.shape
old_vocabulary = set(vectorizer.get_feature_names_out())
old_query_vector = vectorizer.transform(['transformer models'])
documents.append(bert_document)
document_vectors = vectorizer.fit_transform(documents)
new_vocabulary = set(vectorizer.get_feature_names_out())
print('Previous matrix shape:', old_shape)
print('Rebuilt matrix shape:', document_vectors.shape)
print('New vocabulary terms:', sorted(new_vocabulary - old_vocabulary))
print('Query nonzero terms before rebuilding:', old_query_vector.nnz)
bert_results = search('transformer models', k=2)

Previous matrix shape: (6, 39)
Rebuilt matrix shape: (7, 45)
New vocabulary terms: ['bert', 'bidirectional', 'context', 'model', 'transformer', 'uses']
Query nonzero terms before rebuilding: 0
Query: 'transformer models' | k=2
Document 6 | Score: 0.408 | BERT is a transformer model that uses bidirectional context.
Document 0 | Score: 0.000 | Transformers use self-attention to understand relationships between words.


### Question 2 Answer

The new BERT note, document **6**, ranks first. Rebuilding adds its vector, updates the vocabulary and IDF weights, and keeps queries and documents in the same feature space; the old matrix had only six documents.

## Verification

In [7]:
assert len(documents) == 7 and documents[-1] == bert_document
assert old_shape[0] == 6 and document_vectors.shape[0] == 7
assert document_vectors.shape[1] == len(vectorizer.get_feature_names_out())
assert old_query_vector.nnz == 0
assert 'transformer' in new_vocabulary - old_vocabulary
assert similarity_results[0]['document_id'] == 5
assert ner_results[0]['document_id'] == 3 and ner_results[0]['score'] > 0
assert bert_results[0]['document_id'] == 6 and bert_results[0]['score'] > 0
assert all(result['score'] == 0 for result in unrelated_results)
assert filtered_unrelated == []
assert len(filtered_relevant) == 1 and filtered_relevant[0]['document_id'] == 5
assert all(result['score'] > 0 for result in filtered_relevant)
assert np.isfinite(document_vectors.data).all()
assert np.allclose(np.asarray(document_vectors.multiply(document_vectors).sum(axis=1)).ravel(), 1)
assert all(bert_results[i]['score'] >= bert_results[i + 1]['score'] for i in range(len(bert_results) - 1))
print('All search, zero-score filtering, ranking and rebuilt-index checks passed.')

All search, zero-score filtering, ranking and rebuilt-index checks passed.